In [126]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import scipy.stats as sts
from statsmodels.formula.api import ols
import statsmodels.api as sm

In [127]:
data = pd.read_csv('../data/final.csv')
data['log_scored_by'] = np.log10(data['scored_by'])
data['sqrt_chapters'] = np.power(data['chapters'], 0.5)
columns = ['A', 'log_scored_by', 'volumes', 'sqrt_chapters',  'has Romance', 'has Comedy', 'has Hentai', 'has Fantasy',
       'has Boys Love', 'has School', 'has Historical', 'has Harem',
       'has Psychological', 'has Isekai']

In [128]:
def load_YX(columns):
  X = data[columns[1:]].to_numpy()

  X = np.hstack((np.ones((X.shape[0], 1)), X))
  Y = data[['score']].to_numpy()
  return Y, X

Y, X = load_YX(columns)

In [129]:
def solve_multi_linear_model(Y, X):
  B = np.linalg.inv(X.T @ X) @ X.T @ Y
  e = Y - X @ B
  return B, e

B, e = solve_multi_linear_model(Y, X)


In [130]:
def get_corr(columns):
  return data[['score'] + columns[1:]].corr().round(2)

get_corr(columns)

,score,log_scored_by,volumes,sqrt_chapters,has Romance,has Comedy,has Hentai,has Fantasy,has Boys Love,has School,has Historical,has Harem,has Psychological,has Isekai
score,1.00,0.47,0.27,0.38,0.05,0.01,-0.11,0.01,-0.13,0.08,0.01,-0.04,0.01,0.06
log_scored_by,0.47,1.00,0.35,0.35,0.06,0.17,-0.13,-0.02,-0.15,0.15,-0.04,-0.08,0.02,-0.05
volumes,0.27,0.35,1.00,0.87,0.18,0.16,-0.12,0.22,-0.23,0.17,-0.01,0.10,-0.04,-0.03
sqrt_chapters,0.38,0.35,0.87,1.00,0.14,0.15,-0.09,0.21,-0.33,0.21,0.03,0.10,-0.02,0.08
has Romance,0.05,0.06,0.18,0.14,1.00,0.19,-0.15,-0.13,-0.36,0.09,0.04,0.06,0.13,-0.05
has Comedy,0.01,0.17,0.16,0.15,0.19,1.00,-0.12,0.04,-0.09,0.05,-0.16,-0.04,-0.04,-0.05
has Hentai,-0.11,-0.13,-0.12,-0.09,-0.15,-0.12,1.00,-0.05,-0.11,-0.08,-0.05,0.28,-0.05,-0.02
has Fantasy,0.01,-0.02,0.22,0.21,-0.13,0.04,-0.05,1.00,-0.27,-0.07,0.22,0.04,-0.14,0.13
has Boys Love,-0.13,-0.15,-0.23,-0.33,-0.36,-0.09,-0.11,-0.27,1.00,-0.05,-0.09,-0.10,-0.13,-0.04
has School,0.08,0.15,0.17,0.21,0.09,0.05,-0.08,-0.07,-0.05,1.00,-0.10,0.11,-0.10,-0.03


In [131]:
def standart_coef(Y, X, B):
  sigma_y = np.sum((Y - np.mean(Y)) ** 2)
  sigma_x = np.sqrt(np.array([np.sum((X - np.mean(X, axis=0)) ** 2, axis=0)]) / X.shape[0]) 
  return (B[1:] * sigma_y) / sigma_x.T[1:]

def print_cool_B(B):
  ret = f"y = {B[0][0]:.4f} "
  for i in range(1, B.shape[0]):
    ret += f"+ {B[i][0]:.4f} x_{i} "
  return ret

def print_cool_B_norm(B_norm):
  ret = f"t_y = {B_norm[0][0]:.2f} t_x_1 "
  for i in range(1, B_norm.shape[0]):
    ret += f"+ {B_norm[i][0]:.2f} t_x_{i + 1} "
  return ret

B_norm = standart_coef(Y, X, B)
print(print_cool_B(B))
print(print_cool_B_norm(B_norm))


y = 5.7793 + 0.4452 x_1 + -0.0383 x_2 + 0.0869 x_3 + 0.0312 x_4 + -0.1176 x_5 + -0.1626 x_6 + -0.0288 x_7 + 0.0183 x_8 + -0.0737 x_9 + -0.0376 x_10 + -0.0506 x_11 + -0.0321 x_12 + 0.1710 x_13 
t_y = 50.78 t_x_1 + -0.48 t_x_2 + 1.46 t_x_3 + 3.91 t_x_4 + -15.72 t_x_5 + -50.07 t_x_6 + -4.02 t_x_7 + 2.62 t_x_8 + -12.91 t_x_9 + -8.97 t_x_10 + -16.78 t_x_11 + -8.34 t_x_12 + 136.96 t_x_13 


In [132]:
def pretty_table(data):
  p = []
  for line in data:
    while len(p) < len(line):
      p.append(0)
    for i in range(len(line)):
      p[i] = max(p[i], len(line[i]))
  
  def print_line(line):
    def add_spaces(s, c):
      return (" " * c) + s
    ret = add_spaces(line[0], p[0] - len(line[0]))
    for i in range(1, len(line)):
      ret += " │ " + add_spaces(line[i], p[i] - len(line[i]))
    for i in range(len(line), len(p)):
      ret += " │ " + p[i] * " "
    return ret

  line_sep = "─" * p[0]
  for i in range(1, len(p)):
    line_sep += "─┼─" + "─" * p[i]

  ret = ""
  ret += print_line(data[0]) + "\n" + line_sep + '\n'
  for i in range(1, len(data)):
    ret += print_line(data[i]) + "\n"
  return ret


In [133]:
def t_Student_criterion(Y, X, B, e, columns, is_print = True, alpha = 0.05):
  dfd = X.shape[0] - X.shape[1]  # Степени свободы знаменателя (внутригрупповые)

  S_2 = e.T @ e / (dfd) # стандартная ошибка в квадрате
  Cov_B = S_2 * np.linalg.inv(X.T @ X)

  SE = np.sqrt(S_2 * np.array([[Cov_B[i][i]] for i in range(Cov_B.shape[1]) ]))

  t_fact = B / SE

  t_table = sts.t.ppf(1 - alpha / 2, dfd)


  table = [["Название критерия", "B", "t_fact", "B!=0"]]

  ret = []

  for i in range(len(columns)):
    table.append([
      columns[i],
      f"{B[i][0]:.5f}",
      f"{t_fact[i][0]:.3f}",
      "+" if np.abs(t_fact[i][0]) > t_table else "-"
    ])
    ret.append(np.abs(t_fact[i][0]) > t_table)
  
  if is_print:
    print(f"t_table = {t_table:.2f}")
    print()
    print(pretty_table(table))
  return ret, t_fact

t_Student_criterion(Y, X, B, e, columns)
pass

t_table = 1.97

Название критерия │        B │ t_fact │ B!=0
──────────────────┼──────────┼────────┼─────
                A │  5.77929 │ 50.656 │    +
    log_scored_by │  0.44520 │ 11.282 │    +
          volumes │ -0.03829 │ -4.485 │    +
    sqrt_chapters │  0.08686 │  7.515 │    +
      has Romance │  0.03123 │  0.670 │    -
       has Comedy │ -0.11756 │ -2.658 │    +
       has Hentai │ -0.16258 │ -1.531 │    -
      has Fantasy │ -0.02879 │ -0.564 │    -
    has Boys Love │  0.01831 │  0.326 │    -
       has School │ -0.07372 │ -1.282 │    -
   has Historical │ -0.03760 │ -0.475 │    -
        has Harem │ -0.05059 │ -0.457 │    -
has Psychological │ -0.03208 │ -0.381 │    -
       has Isekai │  0.17097 │  0.660 │    -



In [134]:
def F_criterion(Y, X, B, e, alpha = 0.05):
  SS_all = np.sum((Y - np.mean(Y)) ** 2)
  SS_R = np.sum((X @ B - np.mean(Y)) ** 2)
  SS_e = np.sum(e ** 2)
  r = np.sqrt(SS_R / SS_all)

  dfn = X.shape[1] - 1           # Степени свободы числителя (межгрупповые)
  dfd = X.shape[0] - X.shape[1]  # Степени свободы знаменателя (внутригрупповые)

  MS_R = SS_R / (dfn)
  MS_e = SS_e / (dfd)

  F_fact = MS_R / MS_e

  F_table = sts.f.ppf(1 - alpha, dfn, dfd)


  print(f"SS_all = {SS_all:.3f}, SS_R = {SS_R:.3f}, SS_e = {SS_e:.3f} ")
  print(f"Проверка: SS_R + SS_e = {SS_R + SS_e:.3f}")
  print(f"общий коэффициент корреляции: {r:.3f}")
  print(f"""тестнота связи: {
    "не наблюдается" if r < 0.1 else 
    "слабая" if r < 0.3 else
    "умеренная" if r < 0.5 else
    "заметная" if r < 0.7 else
    "высокая" if r < 0.9 else
    "весьма высокая"
  }""")
  print()
  table = [
    ["", "df", "SS", "MS", "F", "значимость F"],
    ["Регрессия", f"{dfn}", f"{SS_R:.3f}", f"{MS_R:.3f}", f"{F_fact:.3f}", f"{F_table:.3f}"],
    ["Остаток", f"{dfd}", f"{SS_e:.3f}", f"{MS_e:.3f}"],
    ["Итого", f"{dfn + dfd}", f"{SS_all:.3f}"]
  ]
  print(pretty_table(table))
  print()
  print("Линейность наблюдается" if F_fact > F_table else "Нет оснований предпологать линейность")

F_criterion(Y, X, B, e)

SS_all = 59.540, SS_R = 18.349, SS_e = 41.191 
Проверка: SS_R + SS_e = 59.540
общий коэффициент корреляции: 0.555
тестнота связи: заметная

          │  df │     SS │    MS │     F │ значимость F
──────────┼─────┼────────┼───────┼───────┼─────────────
Регрессия │  13 │ 18.349 │ 1.411 │ 5.688 │        1.780
  Остаток │ 166 │ 41.191 │ 0.248 │       │             
    Итого │ 179 │ 59.540 │       │       │             


Линейность наблюдается


In [135]:
columns_clear = columns.copy()

while len(columns_clear) > 0:
  Y, X = load_YX(columns_clear)
  B, e = solve_multi_linear_model(Y, X)
  is_goods, t_fact = t_Student_criterion(Y, X, B, e, columns_clear, False)
  is_out = True
  for is_good in is_goods:
    is_out = is_good and is_out
  if is_out:
    break
  i_del = np.argmin(np.abs(t_fact.T[0]))
  columns_clear.pop(i_del)

Y, X = load_YX(columns_clear)
B, e = solve_multi_linear_model(Y, X)

B_norm = standart_coef(Y, X, B)
print(print_cool_B(B))
print(print_cool_B_norm(B_norm))
print()
t_Student_criterion(Y, X, B, e, columns_clear)
F_criterion(Y, X, B, e)


y = 5.7629 + 0.4489 x_1 + -0.0372 x_2 + 0.0836 x_3 + -0.1032 x_4 
t_y = 51.20 t_x_1 + -0.47 t_x_2 + 1.41 t_x_3 + -13.81 t_x_4 

t_table = 1.97

Название критерия │        B │ t_fact │ B!=0
──────────────────┼──────────┼────────┼─────
                A │  5.76290 │ 60.465 │    +
    log_scored_by │  0.44891 │ 12.232 │    +
          volumes │ -0.03721 │ -4.879 │    +
    sqrt_chapters │  0.08358 │  8.243 │    +
       has Comedy │ -0.10325 │ -2.541 │    +

SS_all = 59.540, SS_R = 17.928, SS_e = 41.612 
Проверка: SS_R + SS_e = 59.540
общий коэффициент корреляции: 0.549
тестнота связи: заметная

          │  df │     SS │    MS │      F │ значимость F
──────────┼─────┼────────┼───────┼────────┼─────────────
Регрессия │   4 │ 17.928 │ 4.482 │ 18.849 │        2.423
  Остаток │ 175 │ 41.612 │ 0.238 │        │             
    Итого │ 179 │ 59.540 │       │        │             


Линейность наблюдается


In [136]:
corr = get_corr(columns_clear)
print(corr)

               score  log_scored_by  volumes  sqrt_chapters  has Comedy
score           1.00           0.47     0.27           0.38        0.01
log_scored_by   0.47           1.00     0.35           0.35        0.17
volumes         0.27           0.35     1.00           0.87        0.16
sqrt_chapters   0.38           0.35     0.87           1.00        0.15
has Comedy      0.01           0.17     0.16           0.15        1.00


In [139]:
columns_very_clear = columns_clear.copy()
try:
  columns_very_clear.remove("volumes")
except:
  pass

Y, X = load_YX(columns_very_clear)
B, e = solve_multi_linear_model(Y, X)

B_norm = standart_coef(Y, X, B)
print(print_cool_B(B))
print(print_cool_B_norm(B_norm))
print()
t_Student_criterion(Y, X, B, e, columns_very_clear)
F_criterion(Y, X, B, e)

y = 5.8627 + 0.4312 x_1 + 0.0416 x_2 + -0.1141 x_3 
t_y = 49.18 t_x_1 + 0.70 t_x_2 + -15.26 t_x_3 

t_table = 1.97

Название критерия │        B │ t_fact │ B!=0
──────────────────┼──────────┼────────┼─────
                A │  5.86270 │ 61.355 │    +
    log_scored_by │  0.43122 │ 11.503 │    +
    sqrt_chapters │  0.04161 │  7.549 │    +
       has Comedy │ -0.11411 │ -2.740 │    +

SS_all = 59.540, SS_R = 16.582, SS_e = 42.958 
Проверка: SS_R + SS_e = 59.540
общий коэффициент корреляции: 0.528
тестнота связи: заметная

          │  df │     SS │    MS │      F │ значимость F
──────────┼─────┼────────┼───────┼────────┼─────────────
Регрессия │   3 │ 16.582 │ 5.527 │ 22.645 │        2.656
  Остаток │ 176 │ 42.958 │ 0.244 │        │             
    Итого │ 179 │ 59.540 │       │        │             


Линейность наблюдается


In [140]:
corr = get_corr(columns_very_clear)
print(corr)

               score  log_scored_by  sqrt_chapters  has Comedy
score           1.00           0.47           0.38        0.01
log_scored_by   0.47           1.00           0.35        0.17
sqrt_chapters   0.38           0.35           1.00        0.15
has Comedy      0.01           0.17           0.15        1.00
